In [0]:
# Databricks notebook source

# ============================================================
# 0. CONFIG PATH
# ============================================================

dbutils.widgets.text("path", "")

config_path = dbutils.widgets.get("path")


# ============================================================
# 1. IMPORTS
# ============================================================

import json

import pyspark.sql.functions as F

from common_utils.logging import get_logger
from common_utils.generic_functions import safe_cast


# ============================================================
# 2. LOGGER
# ============================================================

logger = get_logger("dim_product_gold")


# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info(
    "Reading Gold configuration from %s",
    config_path
)

with open(config_path, "r") as f:
    config = json.load(f)

logger.info(
    "Gold configuration loaded successfully"
)


# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================

source_config = config["source"]
target_config = config["target"]

source_name = source_config["source_name"]

source_table = source_config["source_table"]

target_table = target_config["target_table"]


logger.info(
    "[%s] Source table: %s",
    source_name,
    source_table
)

logger.info(
    "[%s] Target table: %s",
    source_name,
    target_table
)


# ============================================================
# 5. READ SOURCE TABLE
# ============================================================

logger.info(
    "[%s] Reading sales from %s",
    source_name,
    source_table
)

sales_df = spark.read.table(
    source_table
)

logger.info(
    "[%s] sales table loaded successfully",
    source_name
)


# ============================================================
# 6. TRANSFORM PRODUCT
# ============================================================

def transform_product(df, config):

    logger.info(
        "[%s] Starting dim_product transformation",
        source_name
    )


    # --------------------------------------------------------
    # Select required columns
    # --------------------------------------------------------

    df = df.select(
        "product_id",
        "product_name",
        "unit",
        "product_category"
    )


    # --------------------------------------------------------
    # Clean product name
    # --------------------------------------------------------

    df = df.withColumn(
        "product_name",
        F.trim(
            F.col("product_name")
        )
    )


    # --------------------------------------------------------
    # Generate deterministic product ID where missing
    #
    # sha2(product_name) gives the same ID every time for
    # the same product name.
    # --------------------------------------------------------

    generated_id = F.concat(
        F.lit(
            config["transform"]["generated_product_id_prefix"]
        ),
        F.substring(
            F.sha2(
                F.coalesce(
                    F.col("product_name"),
                    F.lit("UNKNOWN")
                ),
                256
            ),
            1,
            16
        )
    )


    df = df.withColumn(
        "product_id",
        F.when(
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == ""),
            generated_id
        )
        .otherwise(
            F.col("product_id")
        )
    )


    # --------------------------------------------------------
    # Category
    # --------------------------------------------------------

    df = df.withColumn(
        "category",
        F.coalesce(
            F.col("product_category"),
            F.lit(
                config["transform"]["default_category"]
            )
        )
    )


    # --------------------------------------------------------
    # Unit
    # --------------------------------------------------------

    df = df.withColumn(
        "unit",
        F.coalesce(
            F.col("unit"),
            F.lit(
                config["transform"]["default_unit"]
            )
        )
    )


    # --------------------------------------------------------
    # Cast final columns
    # --------------------------------------------------------

    df = safe_cast(
        df,
        config.get("cast_columns", {})
    )


    # --------------------------------------------------------
    # Select final columns
    # --------------------------------------------------------

    df = df.select(
        *config["select_columns"]
    )


    # --------------------------------------------------------
    # One row per product
    # --------------------------------------------------------

    df = (
        df
        .dropDuplicates(
            ["product_id"]
        )
    )


    logger.info(
        "[%s] dim_product transformation completed",
        source_name
    )

    return df


# ============================================================
# 7. CREATE DIM PRODUCT
# ============================================================

logger.info(
    "[%s] Starting dim_product creation",
    source_name
)

dim_product = transform_product(
    sales_df,
    config
)


# ============================================================
# 8. VALIDATE DATA
# ============================================================

logger.info(
    "[%s] Validating dim_product",
    source_name
)

record_count = dim_product.count()

logger.info(
    "[%s] dim_product record count: %s",
    source_name,
    record_count
)

if record_count == 0:

    raise ValueError(
        "dim_product contains zero records"
    )


duplicate_count = (
    dim_product
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


if duplicate_count > 0:

    raise ValueError(
        "Duplicate product_id detected in dim_product"
    )


# ============================================================
# 9. DISPLAY
# ============================================================

dim_product.show()


# ============================================================
# 10. CREATE GOLD SCHEMA
# ============================================================

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    retaildataplatform.gold
""")


# ============================================================
# 11. WRITE GOLD TABLE
# ============================================================

logger.info(
    "[%s] Writing dim_product to %s",
    source_name,
    target_table
)

(
    dim_product
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(target_table)
)


logger.info(
    "[%s] Gold table %s written successfully",
    source_name,
    target_table
)


# ============================================================
# 12. FINAL VALIDATION
# ============================================================

final_count = (
    spark.read
    .table(target_table)
    .count()
)

logger.info(
    "[%s] Final dim_product record count: %s",
    source_name,
    final_count
)

logger.info(
    "[%s] dim_product pipeline completed successfully",
    source_name
)